In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import requests

In [2]:
df = pd.read_pickle(r"E:\Data Science\Projects\AI Teaching Assistant\new_embeddings.pkl")

OLLAMA_URL = "http://localhost:11434/api/embed"
MODEL_NAME = "bge-m3"
BATCH_SIZE = 16

In [3]:
def create_embedding(text_list):
    r = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "input": text_list
        },
        timeout=300
    )

    r.raise_for_status()

    response = r.json()

    if "embeddings" not in response:
        raise Exception(
            f"Embeddings not found in response.\nResponse: {response}"
        )

    return response["embeddings"]

In [4]:
question = input("Enter your question: ")
question_embedding = create_embedding([question])[0]

In [5]:
similarities = cosine_similarity(np.vstack(df['embedding'].values), [question_embedding]).flatten()
print(f"For question '{question}', similarities with existing embeddings: {similarities}")
print(f"Indices of top 5 most similar embeddings: {similarities.argsort()[-5:][::-1]}")  # Print indices of top 5 most similar embeddings

For question 'Where is perceptron taught in this course?', similarities with existing embeddings: [0.39543355 0.42394361 0.39861879 ... 0.44657213 0.41372891 0.30737417]
Indices of top 5 most similar embeddings: [ 699  680 1444  954  677]


In [6]:
df.iloc[similarities.argsort()[-5:][::-1]]  # Display the top 5 most similar embeddings

,number,title,start,end,text,chunk_id,embedding
699,4,What is a Perceptron | Perceptron Vs Neuron | ...,878.0,888.0,Pause the video once. Just make sure you unde...,699,"[-0.0012164651, -0.016997864, -0.022884574, 0...."
680,4,What is a Perceptron | Perceptron Vs Neuron | ...,651.0,663.0,Now you have a new student. His GPA is 5.1. A...,680,"[-0.019792162, -0.014087713, -0.004487369, -0...."
1444,9,Multi Layer Perceptron | MLP Intuition,1044.0,1065.0,"So, for every student here, I will repeat the...",1444,"[0.0024261302, 0.004985111, -0.032663308, 0.02..."
954,5,Perceptron Trick | How to train a Perceptron |...,1506.0,1516.0,code perceptron trick and then we will code i...,954,"[-0.021379953, 0.0021796476, 0.014032428, 0.00..."
677,4,What is a Perceptron | Perceptron Vs Neuron | ...,618.0,628.0,How to train the perceptron? We will study al...,677,"[-0.0066892374, 0.00035695705, -0.036022622, -..."


In [7]:
new_df = df.iloc[similarities.argsort()[-5:][::-1]]
dff = new_df[['number', 'title', 'start', 'end', 'text']]

In [11]:
prompt = f"""
You are an AI assistant helping students navigate a Deep Learning course.

Below are video transcript chunks. Each chunk contains:
- video_number
- video_title
- start (seconds)
- end (seconds)
- transcript text

Video Chunks:
{dff.to_json(orient="records")}

--------------------------------------------------

Student Question:
"{question}"

Your task:

1. Find all transcript chunks that are relevant to the student's question.
2. Match using semantic meaning, not just exact keyword matching.
   - Include synonyms and related concepts whenever appropriate.
3. Group results by video.
4. For each relevant video, provide:
   - Video Number
   - Video Title
   - Timestamp in MM:SS format
   - A very short description (5–15 words) of what is explained there.
5. Rank results from most relevant to least relevant.
6. If multiple nearby chunks belong to the same explanation, merge them into one timestamp range.
7. If only a brief mention exists, explicitly state "Brief mention".
8. If a concept is explained in depth across multiple timestamps in the same video, mention each important section.

Output format:

Your question, "{question}" is covered in the following videos:

Video <number> – <title>
• MM:SS — <short description with your own words dont just copy the transcript>

Video <number> – <title>
• MM:SS — <short description with your own words dont just copy the transcript>

continue for all relevant videos.

Rules:
- Keep the response under 120 words.
- Do not use end time in the output.
- Do not invent timestamps or explanations.
- Only use the provided transcript chunks.
- Convert starting seconds to MM:SS format.
- If no relevant content exists, respond exactly:
No content found
- Do not ask follow-up questions.
- Do not add introductory or concluding text.
"""

promptt = f"""
Video transcript chunks:
{dff.to_json(orient="records")}

Question: "{question}"

Find all chunks relevant to the question (semantic match, not just keywords).

Return only:
The Relevant Videos for your question, "{question}"
Video <number> - <title>
• MM:SS — <short description with your own words dont just copy the transcript>


Group nearby chunks from the same video. Sort by relevance. Convert starting seconds to MM:SS. Use only the given data. If no relevant chunk exists, reply exactly:
No content found, do not use words like chunk, section, or part instead use word "video". Do not invent timestamps or explanations.
"""

In [12]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def get_api_key():
    """Get API key from environment variables"""
    # Try GEMINI_API_KEY first
    api_key = os.getenv("GEMINI_API_KEY")
    if api_key:
        return api_key
    return None

api_key = get_api_key()
if not api_key:
    raise ValueError("API key not found. Please check your .env file.")

In [15]:
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash') 
model.start_chat(history=[])
# Send message to the model
response = model.generate_content(prompt)
print(response.text)

Your question, "Where is perceptron taught in this course?" is covered in the following videos:

Video 4 – What is a Perceptron | Perceptron Vs Neuron | Perceptron Geometric Intuition
• 09:38 — Understand the perceptron's fundamental concept and inputs.
• 10:18 — Introduction to how the perceptron is trained.
• 10:51 — An example demonstrating the application of a perceptron.

Video 5 – Perceptron Trick | How to train a Perceptron | Perceptron Part 2 | Deep Learning Full Course
• 25:06 — Explains the Perceptron Trick algorithm for training.

Video 9 – Multi Layer Perceptron | MLP Intuition
• 17:24 — Brief mention of a component working as a perceptron within an MLP.
